In [2]:
#!pip install cobaya classy matplotlib numpy clik

  Using cached cobaya-3.6.2-py3-none-any.whl.metadata (9.4 kB)
  Using cached classy-3.3.4.0.tar.gz (6.0 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.6-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached clik-0.92.4.tar.gz (12 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached Py_BOBYQA-1.5.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached getdist-1.7.7-py3-none-any.whl.metadata (13 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached setupt

In [10]:
!./install_planck.sh 

=== Installing Planck likelihood package with Cobaya ===
[install] Installing external packages at '/workspaces/cosmo-codespace/cobaya_packages'
[install] The installation path has been written into the global config file: /home/vscode/.config/cobaya/config.yaml

planck_2018_highl_plik.TTTEEE

[install] Checking if dependencies have already been installed...
[install] Check found no existing installation
[install] (If you expected this to be already installed, re-run `cobaya-install` with --debug to get more verbose output.)
[install] Installing...
[planck_2018_highl_plik.TTTEEE] Installing clipy.
[planck_2018_highl_plik.TTTEEE] Installing pre-requisites...
[planck_2018_highl_plik.TTTEEE] Installing clipy from GitHub repository...
34.8kiB [00:00, 51.3MiB/s]
[planck_2018_highl_plik.TTTEEE] Downloaded filename clipy-clipy_0.15.tar.gz
[planck_2018_highl_plik.TTTEEE] clipy clipy_0.15 downloaded and decompressed correctly.
[planck_2018_highl_plik.TTTEEE] clipy installation finished!
[instal

In [ ]:
import os
os.environ["COBAYA_PACKAGES_PATH"] = "./cobaya_packages"

# You can even run the installer directly from a notebook cell like this:
#!cobaya-install planck_2018_highl_plik.TTTEEE -p ./cobaya_packages


# Initail Graphs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cobaya.model import get_model

# ---------------------------------------------------------------------
# COBAYA INFO (correct structure)
# ---------------------------------------------------------------------

info_th = {
    "params": {
        "omega_b": 0.02237,
        "omega_cdm": 0.1200,
        "H0": 67.36,
        "tau_reio": 0.0544,
        "A_s": 2.10e-9,
        "n_s": 0.9649,
    },

    "theory": {
        "classy": {
            "extra_args": {
                "non_linear": "halofit",
                "l_max_scalars": 2500,
                "output": "tCl pCl lCl"
            }
        }
    },

    "likelihood": {
        #"planck_2018_highl_plik.TTTEEE": None, # Uncommented this likelihood
        "one": { # The dummy likelihood, now correctly specifying its requirements
            "requires": {"Cl": {"tt": 2500, "te": 2500, "ee": 2500}}}
    }
}

print("Initializing Cobaya + CLASS...")
model_th = get_model(info_th)

print("Running evaluation...")
model_th.logposterior({})

# ---------------------------------------------------------------------
# GET Cl CORRECTLY (NO PRIVATE STATE ACCESS)
# ---------------------------------------------------------------------

cl_th = model_th.provider.get_Cl(ell_factor=True, units="muK2")

In [ ]:

# ---------------------------------------------------------------------
# Convert to D_ell
# ---------------------------------------------------------------------

T_cmb = 2.7255e6  # micro-Kelvin

ells = cl_th["ell"]

# Assuming cl already contains D_ell in muK^2:
DlTT = cl_th["tt"]
DlTE = cl_th["te"]
DlEE = cl_th["ee"]

print("✅ Success: CMB spectra computed via Cobaya → CLASS")

# ---------------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------------

fig, axes = plt.subplots(3, 1, figsize=(9, 11), sharex=True)

axes[0].plot(ells[2:], DlTT[2:], color='crimson', lw=2)
axes[0].set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
axes[0].grid(True, alpha=0.4)
axes[0].set_title("ΛCDM CMB spectra (Cobaya + CLASS)")

axes[1].plot(ells[2:], DlTE[2:], color='darkgreen', lw=2)
axes[1].set_ylabel(r'$D_\ell^{TE}\ [\mu K^2]$')
axes[1].grid(True, alpha=0.4)

axes[2].plot(ells[2:], DlEE[2:], color='purple', lw=2)
axes[2].set_ylabel(r'$D_\ell^{EE}\ [\mu K^2]$')
axes[2].set_xlabel(r'Multipole $\ell$')
axes[2].grid(True, alpha=0.4)
axes[2].set_xlim(2, 2500)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cobaya.model import get_model

# ---------------------------------------------------------------------
# 1. COBAYA CONFIGURATION (Theory Only, Absolute Parameters)
# ---------------------------------------------------------------------
info = {
    "params": {
        "omega_b": 0.02237,
        "omega_cdm": 0.1200,
        "H0": 67.36,
        "tau_reio": 0.0544,
        "A_s": 2.10e-9,
        "n_s": 0.9649,

        # Nuisance parameters for Planck high-l TTTEEE likelihood
        "a_ini": {
            "prior": {"min": 0.5, "max": 2.5},
            "ref": 1.0,
            "proposal": 0.1
        },
        "A_CIB_217": {
            "prior": {"min": 0, "max": 10},
            "ref": 6.0,
            "proposal": 0.5
        },
        "xi_sz_cib": {
            "prior": {"min": 0, "max": 1},
            "ref": 0.3,
            "proposal": 0.05
        },
        "r_d": {
            "prior": {"min": 0, "max": 10},
            "ref": 0.9,
            "proposal": 0.1
        },
        "calib_100_tt": {
            "prior": {"min": 0.9, "max": 1.1},
            "ref": 1.0,
            "proposal": 0.01
        },
        "calib_217_tt": {
            "prior": {"min": 0.9, "max": 1.1},
            "ref": 1.0,
            "proposal": 0.01
        },
        # Additional common Planck high-l nuisance parameters
        "a_ps_100": {"prior": {"min": 0, "max": 200}, "ref": 100, "proposal": 10},
        "a_ps_143": {"prior": {"min": 0, "max": 200}, "ref": 100, "proposal": 10},
        "a_ps_217": {"prior": {"min": 0, "max": 200}, "ref": 100, "proposal": 10},
        "a_cib_143": {"prior": {"min": 0, "max": 20}, "ref": 10, "proposal": 1},
        "a_cib_217": {"prior": {"min": 0, "max": 40}, "ref": 20, "proposal": 2},
        "a_ksz": {"prior": {"min": 0, "max": 10}, "ref": 3, "proposal": 0.5},
        "a_tsz": {"prior": {"min": 0, "max": 10}, "ref": 3, "proposal": 0.5},
        "ps_calib_143": {"prior": {"min": 0.9, "max": 1.1}, "ref": 1.0, "proposal": 0.01},
        "ps_calib_217": {"prior": {"min": 0.9, "max": 1.1}, "ref": 1.0, "proposal": 0.01}
    },

    "theory": {
        "classy": {
            "extra_args": {
                "non_linear": "halofit",
                "l_max_scalars": 2500,
                "output": "tCl pCl lCl"
            }
        }
    },

    # The clean dummy likelihood that keeps Cobaya's graph engine happy
    "likelihood": {
        "planck_2018_highl_plik.TTTEEE": {
            "requires": {"Cl": {"tt": 2500, "te": 2500, "ee": 2500}}
        }
    }
}

print("Initializing Cobaya + CLASS Engine...")
model = get_model(info)

print("Running baseline cosmological forward evaluation...")
#model.logposterior({})

#cl = model.provider.get_Cl() # Re-adding this line

# ---------------------------------------------------------------------
# 2. EXTRACT THEORY DATA CLEANLY (via Public Interface)
# ---------------------------------------------------------------------
ells_theory = cl["ell"]

T_cmb = 2.7255e6  # micro-Kelvin
factor = ells_theory * (ells_theory + 1) / (2 * np.pi)

DlTT = factor * cl["tt"] * (T_cmb**2)
DlTE = factor * cl["te"] * (T_cmb**2)
DlEE = factor * cl["ee"] * (T_cmb**2)

# ---------------------------------------------------------------------
# 3. EXTRACT EXPERIMENTAL PLANCK DATA POINTS & ERRORS
# ---------------------------------------------------------------------
print("Extracting experimental data windows...")
like_instance = model.likelihood["planck_2018_highl_plik.TTTEEE"]

data_vector = like_instance.data.vector
covariance_matrix = like_instance.data.covariance
experimental_errors = np.sqrt(np.diag(covariance_matrix))
ells_binned = like_instance.data.bin_centers

# Slicing the binned data vector into the standard Planck splits:
# TT: 25 bins | TE: 9 bins | EE: 27 bins
obs_tt = data_vector[0:25]
err_tt = experimental_errors[0:25]
ells_tt = ells_binned[0:25]

obs_te = data_vector[25:34]
err_te = experimental_errors[25:34]
ells_te = ells_binned[25:34]

obs_ee = data_vector[34:61]
err_ee = experimental_errors[34:61]
ells_ee = ells_binned[34:61]

print("✅ Success: All data and theory arrays ready.")

# ---------------------------------------------------------------------
# 4. PLOTTING BOTH CHANNELS TOGETHER
# ---------------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharex=True)
plt.subplots_adjust(hspace=0.15)

# --- TT Panel ---
axes[0].plot(ells_theory[2:], DlTT[2:], color='crimson', lw=2, label=r'$\Lambda$CDM Theory Baseline')
axes[0].errorbar(ells_tt, obs_tt, yerr=err_tt, fmt='o', color='darkblue',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data')
axes[0].set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$', fontsize=12)
axes[0].grid(True, alpha=0.3, linestyle=':')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_title("Planck 2018 High-$\ell$ Observations vs. Boltzmann Theory", fontsize=14, pad=15)

# --- TE Panel ---
axes[1].plot(ells_theory[2:], DlTE[2:], color='darkgreen', lw=2)
axes[1].errorbar(ells_te, obs_te, yerr=err_te, fmt='o', color='forestgreen',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8)
axes[1].set_ylabel(r'$D_\ell^{TE}\ [\mu K^2]$', fontsize=12)
axes[1].grid(True, alpha=0.3, linestyle=':')

# --- EE Panel ---
axes[2].plot(ells_theory[2:], DlEE[2:], color='purple', lw=2)
axes[2].errorbar(ells_ee, obs_ee, yerr=err_ee, fmt='o', color='indigo',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8)
axes[2].set_ylabel(r'$D_\ell^{EE}\ [\mu K^2]$', fontsize=12)
axes[2].set_xlabel(r'Multipole Moment ($\ell$)', fontsize=12)
axes[2].set_xlim(2, 2500)
axes[2].grid(True, alpha=0.3, linestyle=':')

plt.tight_layout()
plt.show()

# Plot the TT, EE and TE graphs ...  

In [ ]:
import numpy as np

# 1. Grab the active likelihood instance from your model
like_instance = model.likelihood["planck_2018_highl_plik.TTTEEE"]
internal_lkl = like_instance.clik_likelihood._internal

# 2. Extract the unbinned data vector and the pre-loaded inverse covariance
obs_vector = np.array(internal_lkl.data_bandpowers)
inv_covariance_matrix = np.array(internal_lkl.siginv)

# 3. Manually invert the precision matrix to get the true covariance matrix
covariance_matrix = np.linalg.inv(inv_covariance_matrix)

# 4. Calculate the standard 1-sigma uncertainties from the diagonal
experimental_errors = np.sqrt(np.diag(covariance_matrix))

# 5. Segment into equal thirds for your TT, TE, and EE channel subplots
num_points = len(obs_vector)
segment = num_points // 3

obs_tt = obs_vector[0:segment]
err_tt = experimental_errors[0:segment]
ells_tt = np.linspace(30, 2500, len(obs_tt))

obs_te = obs_vector[segment:2*segment]
err_te = experimental_errors[segment:2*segment]
ells_te = np.linspace(30, 2500, len(obs_te))

obs_ee = obs_vector[2*segment:]
err_ee = experimental_errors[2*segment:]
ells_ee = np.linspace(30, 2500, len(obs_ee))

print(f"Data Vector Length: {num_points}")
print(f"Inverse Covariance Shape: {inv_covariance_matrix.shape}")
print(f"TT points: {len(obs_tt)}, TE points: {len(obs_te)}, EE points: {len(obs_ee)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np # Import numpy as it will be used for linspace

# ---------------------------------------------------------------------
# PLOTTING EXPERIMENTAL DATA
# ---------------------------------------------------------------------

# Re-evaluate ells_xx to ensure they match the length of obs_xx and err_xx
# This addresses the ValueError: 'x' and 'y' must have the same size
# if ells_xx somehow got corrupted or wasn't updated correctly.
ells_tt = np.linspace(30, 2500, len(obs_tt))
ells_te = np.linspace(30, 2500, len(obs_te))
ells_ee = np.linspace(30, 2500, len(obs_ee))

fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharex=True)
plt.subplots_adjust(hspace=0.15)

# --- TT Panel ---
axes[0].errorbar(ells_tt, obs_tt, yerr=err_tt, fmt='o', color='darkblue',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data TT')
axes[0].set_ylabel(r'$D_{\ell}^{TT}\text{ [}\mu\text{K}^2]$', fontsize=12)
axes[0].grid(True, alpha=0.3, linestyle=':')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_title(r"Planck 2018 High-$\ell$ Experimental Data", fontsize=14, pad=15)

# --- TE Panel ---
axes[1].errorbar(ells_te, obs_te, yerr=err_te, fmt='o', color='forestgreen',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data TE')
axes[1].set_ylabel(r'$D_{\ell}^{TE}\text{ [}\mu\text{K}^2]$', fontsize=12)
axes[1].grid(True, alpha=0.3, linestyle=':')
axes[1].legend(loc='upper right', fontsize=10)

# --- EE Panel ---
axes[2].errorbar(ells_ee, obs_ee, yerr=err_ee, fmt='o', color='indigo',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data EE')
axes[2].set_ylabel(r'$D_{\ell}^{EE}\text{ [}\mu\text{K}^2]$', fontsize=12)
axes[2].set_xlabel(r'Multipole Moment ($\ell$)', fontsize=12)
axes[2].set_xlim(2, 2500)
axes[2].grid(True, alpha=0.3, linestyle=':')
axes[2].legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:


# ---------------------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------------------

fig, axes = plt.subplots(3, 1, figsize=(9, 11), sharex=True)

# TT Panel
axes[0].plot(ells[2:], DlTT[2:], color='crimson', lw=2, label=r'$\Lambda$CDM Theory Baseline')
axes[0].errorbar(ells_tt, obs_tt, yerr=err_tt, fmt='o', color='darkblue',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data TT')
axes[0].set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
axes[0].grid(True, alpha=0.4)
axes[0].set_title("$\Lambda$CDM CMB spectra (Cobaya + CLASS) vs Planck 2018 Data")
axes[0].legend(loc='upper right')

# TE Panel
axes[1].plot(ells[2:], DlTE[2:], color='darkgreen', lw=2, label=r'$\Lambda$CDM Theory Baseline TE')
axes[1].errorbar(ells_te, obs_te, yerr=err_te, fmt='o', color='forestgreen',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data TE')
axes[1].set_ylabel(r'$D_\ell^{TE}\ [\mu K^2]$')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='upper right')

# EE Panel
axes[2].plot(ells[2:], DlEE[2:], color='purple', lw=2, label=r'$\Lambda$CDM Theory Baseline EE')
axes[2].errorbar(ells_ee, obs_ee, yerr=err_ee, fmt='o', color='indigo',
                 ecolor='gray', markersize=4, capsize=2, elinewidth=1, alpha=0.8, label='Planck 2018 Experimental Data EE')
axes[2].set_ylabel(r'$D_\ell^{EE}\ [\mu K^2]$')
axes[2].set_xlabel(r'Multipole $\ell$')
axes[2].grid(True, alpha=0.4)
axes[2].set_xlim(2, 2500)
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

### Comparison of Experimental and Theoretical Magnitudes

In [ ]:
print("--- TT Channel ---")
print("ell | Experimental Obs | Experimental Err | Theoretical DlTT")
for i in range(min(10, len(ells_tt))):
    # Find the closest theoretical ell for comparison
    closest_ell_idx = np.abs(ells - ells_tt[i]).argmin()
    print(f"{ells_tt[i]:<4.0f} | {obs_tt[i]:<18.4f} | {err_tt[i]:<16.4f} | {DlTT[closest_ell_idx]:<16.4f}")

print("\n--- TE Channel ---")
print("ell | Experimental Obs | Experimental Err | Theoretical DlTE")
for i in range(min(10, len(ells_te))):
    closest_ell_idx = np.abs(ells - ells_te[i]).argmin()
    print(f"{ells_te[i]:<4.0f} | {obs_te[i]:<18.4f} | {err_te[i]:<16.4f} | {DlTE[closest_ell_idx]:<16.4f}")

print("\n--- EE Channel ---")
print("ell | Experimental Obs | Experimental Err | Theoretical DlEE")
for i in range(min(10, len(ells_ee))):
    closest_ell_idx = np.abs(ells - ells_ee[i]).argmin()
    print(f"{ells_ee[i]:<4.0f} | {obs_ee[i]:<18.4f} | {err_ee[i]:<16.4f} | {DlEE[closest_ell_idx]:<16.4f}")

### Ratio of Theory to Experimental, Divided by L(L+1)

In [ ]:
import numpy as np

print("--- TT Channel ---")
print("ell | (Theory Dl / Exp Obs) / (L(L+1))")
for i in range(min(10, len(ells_tt))):
    ell_exp = ells_tt[i]
    obs_exp = obs_tt[i]

    # Find the closest theoretical ell for comparison
    closest_ell_idx = np.abs(ells - ell_exp).argmin()
    ell_theory = ells[closest_ell_idx]
    Dl_theory = DlTT[closest_ell_idx]

    # Calculate the ratio and then divide by L(L+1)
    ratio_Dl = Dl_theory / obs_exp
    l_lplus1 = ell_theory * (ell_theory + 1)

    # Ensure division by zero is handled, though unlikely for ell >= 30
    if l_lplus1 == 0:
        result = np.nan
    else:
        result = ratio_Dl / l_lplus1
    print(f"{ell_exp:<4.0f} | {result:<28.4e}")

print("\n--- TE Channel ---")
print("ell | (Theory Dl / Exp Obs) / (L(L+1))")
for i in range(min(10, len(ells_te))):
    ell_exp = ells_te[i]
    obs_exp = obs_te[i]

    closest_ell_idx = np.abs(ells - ell_exp).argmin()
    ell_theory = ells[closest_ell_idx]
    Dl_theory = DlTE[closest_ell_idx]

    ratio_Dl = Dl_theory / obs_exp
    l_lplus1 = ell_theory * (ell_theory + 1)
    if l_lplus1 == 0:
        result = np.nan
    else:
        result = ratio_Dl / l_lplus1
    print(f"{ell_exp:<4.0f} | {result:<28.4e}")

print("\n--- EE Channel ---")
print("ell | (Theory Dl / Exp Obs) / (L(L+1))")
for i in range(min(10, len(ells_ee))):
    ell_exp = ells_ee[i]
    obs_exp = obs_ee[i]

    closest_ell_idx = np.abs(ells - ell_exp).argmin()
    ell_theory = ells[closest_ell_idx]
    Dl_theory = DlEE[closest_ell_idx]

    ratio_Dl = Dl_theory / obs_exp
    l_lplus1 = ell_theory * (ell_theory + 1)
    if l_lplus1 == 0:
        result = np.nan
    else:
        result = ratio_Dl / l_lplus1
    print(f"{ell_exp:<4.0f} | {result:<28.4e}")

# Running a full MCMC sampler

Now, let's configure Cobaya to run a full MCMC sampler using the `planck_2018_highl_plik.TTTEEE` likelihood. This involves defining priors for the cosmological parameters and setting up the MCMC sampler.

In [28]:
%%time
import os
import cobaya
import shutil

# Ensure CLIK_PATH is set so the Planck likelihood can find the data
os.environ["CLIK_PATH"] = "/workspaces/cosmo-codespace/cobaya_packages/data/planck_2018/baseline"
os.environ["COBAYA_PACKAGES_PATH"] = "./cobaya_packages"

print("CLIK_PATH:", os.environ["CLIK_PATH"])
print("COBAYA_PACKAGES_PATH:", os.environ["COBAYA_PACKAGES_PATH"])

# Use ABSOLUTE path for output directory
output_dir = os.path.abspath("cobaya_mcmc_results")
print(f"Output directory (absolute): {output_dir}")

# Clean up old output to avoid resume conflicts
if os.path.exists(output_dir):
    print(f"Cleaning old output directory: {output_dir}")
    shutil.rmtree(output_dir)

# Create the output directory
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

# Change working directory to the output folder
os.chdir(output_dir)
print(f"Changed working directory to: {os.getcwd()}")

# Construct absolute path to clik file
clik_path = os.path.join(os.environ["CLIK_PATH"], "plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik")
print(f"Using clik_file: {clik_path}")

# Define the info dictionary for the MCMC run
info_mcmc = {
    "params": {
        "omega_b": {
            "prior": {"min": 0.02, "max": 0.025},
            "ref": {"min": 0.022, "max": 0.023},
            "proposal": 0.0001,
            "latex": r"\Omega_b h^2"
        },
        "omega_cdm": {
            "prior": {"min": 0.1, "max": 0.14},
            "ref": {"min": 0.11, "max": 0.13},
            "proposal": 0.001,
            "latex": r"\Omega_{cdm} h^2"
        },
        "H0": {
            "prior": {"min": 60, "max": 75},
            "ref": {"min": 65, "max": 70},
            "proposal": 0.5,
            "latex": r"H_0"
        },
        "tau_reio": {
            "prior": {"min": 0.03, "max": 0.07},
            "ref": {"min": 0.05, "max": 0.06},
            "proposal": 0.001,
            "latex": r"\tau_{reio}"
        },
        "A_s": {
            "prior": {"min": 1.5e-9, "max": 2.5e-9},
            "ref": {"min": 2.0e-9, "max": 2.2e-9},
            "proposal": 1e-10,
            "latex": r"A_s"
        },
        "n_s": {
            "prior": {"min": 0.9, "max": 1.0},
            "ref": {"min": 0.95, "max": 0.97},
            "proposal": 0.001,
            "latex": r"n_s"
        },
    },

    "theory": {
        "classy": {
            "extra_args": {
                "non_linear": "halofit",
                "l_max_scalars": 2500,
                "output": "tCl pCl lCl"
            }
        }
    },

    "likelihood": {
        "planck_2018_highl_plik.TTTEEE": {
            "path": os.environ["CLIK_PATH"],
            "clik_file": clik_path  # Use absolute path
        }
    },

    "sampler": {
        "mcmc": {
            "max_samples": 50
        }
    },
    "output": "mcmc"  # Relative to current working directory (cobaya_mcmc_results)
}

print("Starting MCMC run with Planck likelihood...")
print(f"Output will be saved to: {output_dir}")
print(f"Chains will be written to: {os.path.join(output_dir, 'mcmc')}")
try:
    # Run the MCMC sampler with force=True for a fresh start
    updated_info, products = cobaya.run(info_mcmc, force=True)
    print("✅ MCMC run finished successfully!")
    print(f"Check the output directory for chain files.")
    # List output files in current directory
    import glob
    files = sorted(glob.glob("*"))
    print(f"\nOutput files created ({len(files)} total):")
    for f in files[:20]:
        print(f"  - {f}")
except Exception as e:
    import traceback
    print(f"❌ MCMC run failed with error:\n{type(e).__name__}: {e}")
    traceback.print_exc()



CLIK_PATH: /workspaces/cosmo-codespace/cobaya_packages/data/planck_2018/baseline
COBAYA_PACKAGES_PATH: ./cobaya_packages
Output directory (absolute): /workspaces/cosmo-codespace/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results
Created output directory: /workspaces/cosmo-codespace/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results
Changed working directory to: /workspaces/cosmo-codespace/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results
Using clik_file: /workspaces/cosmo-codespace/cobaya_packages/data/planck_2018/baseline/plc_3.0/hi_l/plik/plik_rd12_HM_v22b_TTTEEE.clik
Starting MCMC run with Planck likelihood...
Output will be saved to: /workspaces/cosmo-codespace/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results
Chains will be written to: /workspaces/cosmo-codespace/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/cobaya_mcmc_results/mcmc
[output] Output to

#ChatGPT suggestins

In [ ]:
get_model(info_mcmc)

model = get_model(info_mcmc)
#model.logposterior({})

#print(model.output)
print(model.provider)

In [ ]:
from getdist.mcsamples import loadMCSamples

samples = loadMCSamples("./cobaya_mcmc_results")

print("Loaded samples:", samples.numrows)
print("Parameters:", len(samples.paramNames.list()))
samples.paramNames.list()

In [ ]:
print(samples.getMeans())
print(samples.getCovMat())

In [ ]:
import pandas as pd
import numpy as np

# `samples` is available from CqE4nneYDNFJ
# Get parameter names
param_names_list = samples.paramNames.list()

# Collect all parameter values into a dictionary, then to DataFrame
samples_data = {}
for name in param_names_list:
    param_idx = param_names_list.index(name)
    samples_data[name] = samples.samples[:, param_idx]

samples_df = pd.DataFrame(samples_data)

# Now find the best-fit: the sample with the minimum chi2 for the Planck likelihood
# Note: 'chi2__planck_2018_highl_plik.TTTEEE' is the name of the chi2 parameter for this likelihood
best = samples_df.iloc[samples_df["chi2__planck_2018_highl_plik.TTTEEE"].idxmin()]
print("Best-fit parameters from MCMC chain:")
print(best)

### Generate a Triangle Plot from MCMC Chains

To generate the triangle plot with your cosmological parameters, please ensure you have re-run the MCMC (cell `c9ed9733`) and loaded the samples (cell `CqE4nneYDNFJ`) first. Then execute the cell below.

In [ ]:
import getdist.plots as gdplt

# Print available parameter names one more time to confirm they are loaded
print("Parameters available in samples for plotting:", samples.getParamNames())

gdplot = gdplt.get_subplot_plotter()

gdplot.triangle_plot(
    samples,
    ["omega_b", "omega_cdm", "H0", "tau_reio", "A_s", "n_s"],
    filled=True
)


In [ ]:
from classy import Class

cosmo = Class()

cosmo.set({
    "omega_b": best["omega_b"],
    "omega_cdm": best["omega_cdm"],
    "H0": best["H0"],
    "A_s": best["A_s"],
    "n_s": best["n_s"],
    "tau_reio": best["tau_reio"],
    "l_max_scalars": 2500,
    'lensing': 'yes',
    "output": "tCl pCl lCl",
})

cosmo.compute()
cl = cosmo.raw_cl(2500)

#Gemini suggstions

In [ ]:
import os
from getdist import mcsamples, plots
import getdist.mcsamples as gdmc

# 1. Define the path where your Cobaya output prefix is located
# Replace 'your_output_prefix' with whatever you set 'output' to in your MCMC config
chain_prefix = "./cobaya_mcmc_results"

print("Loading and processing MCMC chains...")
# loadMCSamples automatically handles burning-in removal and weight processing
samples = gdmc.loadMCSamples(chain_prefix, settings={'ignore_rows': 0.3})

# 2. Print out the formal LaTeX table of your parameter constraints
# The 'MCSamples' object does not have a 'getMCMCresult' method. Direct access to 'samples' is used.
print(samples.getInlineLatex('H0', limit=1))      # 68% upper/lower bounds
print(samples.getInlineLatex('omega_b', limit=1))
print(samples.getInlineLatex('omega_cdm', limit=1))

In [ ]:
# 3. Initialize the triangle plotter
gplot = plots.get_subplot_plotter(width_inch=8)

# Print available parameter names to help diagnose any 'parameter not found' errors
print("Available parameter names from samples:", samples.getParamNames())

# You can load your alternative model chains and the standard baseline chains simultaneously
# to overplot them on top of each other.
gplot.triangle_plot(
    [samples],
    params=['H0', 'omega_b', 'omega_cdm', 'n_s'], # Changed 'ns' to 'n_s'
    filled=True,
    title_limit=1  # Automatically prints the mean and 1-sigma error above the 1D histograms
)

# Export as a vector PDF for your manuscript
#gplot.export('cosmological_constraints.pdf')